# MAPNAI — Agent 4 event model: DistilRoBERTa training (Colab / Kaggle)

Fine-tunes `distilroberta-base` to predict the **event type** of an article (8 classes). Each event type maps to
Agent 4's `event_severity` factor, so this is the main model of the offline risk scorer — see
`AGENT4_RISK_ARCHITECTURE.md`. The TF-IDF + logistic regression fallback is trained alongside, on the same split,
so the two can be compared fairly.

| Step | What it does | Time on one T4 |
|---|---|---|
| 1. Setup | Finds the GPU and the data files | < 1 min |
| 2–3. Settings, taxonomy | Hyper-parameters and the 8 event types | — |
| 4. Data | Loads the LLM labels, holds out the gold set, fixes doping labels, splits train / validation | < 1 min |
| 5. Baseline | TF-IDF + logistic regression on the same split | < 1 min |
| 6–7. Train | DistilRoBERTa, best epoch kept by validation macro-F1 | 10–20 min |
| 8. Calibrate | Temperature scaling, so confidence matches accuracy | < 1 min |
| 9. Validation report | Per-type scores, both models | < 1 min |
| 10. Gold evaluation | Scores both models **and Qwen** on your hand labels (skipped if not labelled yet) | < 1 min |
| 11. Save | `risk_event/` model folder, zipped for download | 1 min |

**Files it needs** (all from `data/risk/` in the MAPNAI repo)

| File | Required? |
|---|---|
| `risk_labeled.jsonl` | Yes — the v2 Kaggle labels |
| `gold_ids.txt` | Yes — keeps the gold articles out of training |
| `gold_to_label.csv` | Optional — once hand-labelled, step 10 scores the models on it |
| `gold_key.csv` | Optional — Qwen's answers for the gold set, to measure Qwen itself |

**Colab:** in Google Drive create **My Drive → `MAPNAI_risk` → `train`** and upload the files there.
**Runtime → Change runtime type → T4 GPU**, then **Runtime → Run all**. The model zip downloads at the end.

**Kaggle:** create a dataset with the files and **Add Input** it. **Settings → Accelerator → GPU T4 x2** (one GPU is
used), then **Save Version → Save & Run All**. The zip is in the run's **Output** tab.

**After training:** unzip `mapnai_risk_event_model.zip` into the repo so that you have `models/risk_event/config.json`.

## 1. Setup

In [ ]:
import glob, json, math, os, random, re, shutil, time
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import torch

ON_KAGGLE = os.path.exists('/kaggle/input')
# Kaggle's image also ships the google.colab package, so check for Kaggle first
ON_COLAB = not ON_KAGGLE and os.path.exists('/var/colab/hostname')

if ON_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    DATA_DIRS = ['/content/drive/MyDrive/MAPNAI_risk/train']
    OUT_DIR = '/content/drive/MyDrive/MAPNAI_risk/train/output'
elif ON_KAGGLE:
    DATA_DIRS = ['/kaggle/input']
    OUT_DIR = '/kaggle/working'
else:  # local run from the repo root
    DATA_DIRS = [os.environ.get('MAPNAI_RISK_DATA', 'data/risk')]
    OUT_DIR = os.environ.get('MAPNAI_RISK_OUT', 'models/risk_train_output')
os.makedirs(OUT_DIR, exist_ok=True)


def find(name):
    # First file called `name` under the data folders (Kaggle nests datasets a few levels deep)
    for d in DATA_DIRS:
        hits = sorted(glob.glob(f'{d}/**/{name}', recursive=True))
        if hits:
            return hits[0]
    return None


LABELED_PATH = find('risk_labeled.jsonl')
GOLD_IDS_PATH = find('gold_ids.txt')
GOLD_CSV_PATH = find('gold_to_label.csv')
GOLD_KEY_PATH = find('gold_key.csv')
assert LABELED_PATH, f'risk_labeled.jsonl not found under {DATA_DIRS} — see "Files it needs" above.'
assert GOLD_IDS_PATH, 'gold_ids.txt not found — without it the gold articles would leak into training.'

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Environment:', 'Colab' if ON_COLAB else 'Kaggle' if ON_KAGGLE else 'local')
print('Device     :', torch.cuda.get_device_name(0) if DEVICE == 'cuda' else 'CPU (slow — enable a GPU)')
for k, v in dict(labels=LABELED_PATH, gold_ids=GOLD_IDS_PATH, gold_csv=GOLD_CSV_PATH, gold_key=GOLD_KEY_PATH).items():
    print(f'{k:9s}:', v)
print('Output     :', OUT_DIR)

## 2. Settings

In [ ]:
MODEL_NAME = 'distilroberta-base'
MAX_LEN = 256          # tokens; title + first ~1,500 characters of the body
MAX_BODY_CHARS = 1500
EPOCHS = 4
LR = 3e-5
BATCH = 32
WARMUP_FRAC = 0.06
WEIGHT_DECAY = 0.01
VAL_FRAC = 0.15
PATIENCE = 2           # stop after this many epochs without a better validation macro-F1
SEED = 42
FIX_INTEGRITY = True   # relabel clear doping / fixing articles that the LLM called controversy, legal or ban
LIMIT = None           # e.g. 300 for a quick test run

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

## 3. Event taxonomy
Must match `config/risk_taxonomy.py` in the repo.

In [ ]:
EVENT_SEVERITY = {
    'routine': 2, 'controversy': 8, 'injury': 12, 'legal': 16,
    'ban_suspension': 17, 'cancellation': 18, 'integrity': 21, 'death_tragedy': 25,
}
EVENT_TYPES = list(EVENT_SEVERITY)
LABEL2ID = {t: i for i, t in enumerate(EVENT_TYPES)}
LEGACY_EVENT_TYPES = {'doping': 'integrity', 'match_fixing': 'integrity'}
HIGH_SEVERITY = 16     # legal and above

# Only unambiguous doping / fixing terms — used by FIX_INTEGRITY
INTEGRITY_RX = (r'\b(?:doping|tested positive|failed (?:a )?(?:drugs? |dope )?test|drugs? tests?|banned substance'
                r'|steroids?|anti-doping|wada|usada|performance.enhancing|match.fixing|spot.fixing|illegal betting)\b')


def event_text(title, body):
    return f"{(title or '').strip()}. {(body or '').strip()[:MAX_BODY_CHARS]}".strip()

## 4. Data

In [ ]:
from sklearn.model_selection import train_test_split

lab = pd.read_json(LABELED_PATH, lines=True)
versions = sorted(lab.prompt_version.dropna().unique().tolist()) if 'prompt_version' in lab else []
print(f'{len(lab)} labelled rows | prompt versions {versions}')
if versions != [2]:
    print('WARNING: expected v2 labels only (the Kaggle run). Older labels use a different taxonomy.')

df = lab[lab.in_scope.astype(bool)].copy()
df['y'] = df.event_type.replace(LEGACY_EVENT_TYPES)
df = df[df.y.isin(EVENT_TYPES)]
gold_ids = set(open(GOLD_IDS_PATH).read().split())
n_before = len(df)
df = df[~df.id.isin(gold_ids)].reset_index(drop=True)
print(f'{n_before} in-scope rows, {n_before - len(df)} gold rows held out -> {len(df)} for training + validation')
df['text'] = [event_text(t, b) for t, b in zip(df.title.fillna(''), df.body.fillna(''))]

if FIX_INTEGRITY:
    fix = df.y.isin(['controversy', 'legal', 'ban_suspension']) & df.text.str.contains(INTEGRITY_RX, case=False)
    print(f'FIX_INTEGRITY: {fix.sum()} rows relabelled to integrity', df.y[fix].value_counts().to_dict())
    df.loc[fix, 'y'] = 'integrity'

if LIMIT:
    df = df.sample(min(LIMIT, len(df)), random_state=SEED).reset_index(drop=True)

stratify = df.y if df.y.value_counts().min() >= 2 else None
train_df, val_df = train_test_split(df, test_size=VAL_FRAC, random_state=SEED, stratify=stratify)
train_df, val_df = train_df.reset_index(drop=True), val_df.reset_index(drop=True)

counts = pd.DataFrame({'train': train_df.y.value_counts(), 'val': val_df.y.value_counts()}).reindex(EVENT_TYPES)
print(counts.fillna(0).astype(int).to_string())
for t, n in counts.train.fillna(0).items():
    if n < 50 and not LIMIT:
        print(f'WARNING: only {int(n)} training examples for {t}')

## 5. Baseline — TF-IDF + logistic regression (Agent 4's fallback)

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, f1_score
from sklearn.pipeline import make_pipeline

tfidf = make_pipeline(
    TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True, max_features=200_000),
    LogisticRegression(max_iter=3000, C=4, class_weight='balanced'),
).fit(train_df.text, train_df.y)
tfidf_val_pred = tfidf.predict(val_df.text)


def summary(y_true, y_pred):
    y_true, y_pred = list(y_true), list(y_pred)
    sev_t = np.array([EVENT_SEVERITY[t] for t in y_true])
    sev_p = np.array([EVENT_SEVERITY[t] for t in y_pred])
    hi = sev_t >= HIGH_SEVERITY
    return {
        'accuracy': round(accuracy_score(y_true, y_pred), 4),
        'macro_f1': round(f1_score(y_true, y_pred, labels=EVENT_TYPES, average='macro', zero_division=0), 4),
        'high_sev_recall': round(float(((sev_p >= HIGH_SEVERITY) & hi).sum() / max(1, hi.sum())), 4),
        'severity_mae': round(float(np.abs(sev_t - sev_p).mean()), 3),
    }


print('TF-IDF validation:', summary(val_df.y, tfidf_val_pred))

## 6. DistilRoBERTa — model and data loaders

In [ ]:
from torch.utils.data import DataLoader
from transformers import AutoModelForSequenceClassification, AutoTokenizer, get_linear_schedule_with_warmup

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=len(EVENT_TYPES),
    id2label={i: t for t, i in LABEL2ID.items()}, label2id=LABEL2ID,
).to(DEVICE)


def make_loader(frame, shuffle):
    items = list(zip(frame.text, frame.y.map(LABEL2ID)))

    def collate(batch):
        enc = tokenizer([t for t, _ in batch], truncation=True, max_length=MAX_LEN, padding=True,
                        return_tensors='pt')
        enc['labels'] = torch.tensor([y for _, y in batch])
        return enc

    return DataLoader(items, batch_size=BATCH if shuffle else BATCH * 2, shuffle=shuffle, collate_fn=collate)


train_loader, val_loader = make_loader(train_df, True), make_loader(val_df, False)

# sqrt inverse-frequency class weights: rare types matter more, without swamping routine
freq = train_df.y.value_counts().reindex(EVENT_TYPES).fillna(0).to_numpy() + 1
w = np.sqrt(freq.sum() / freq)
class_weights = torch.tensor(w / w.mean(), dtype=torch.float, device=DEVICE)
print('class weights:', dict(zip(EVENT_TYPES, class_weights.cpu().numpy().round(2))))

## 7. Train
Keeps the epoch with the best validation macro-F1.

In [ ]:
@torch.no_grad()
def predict_logits(loader):
    model.eval()
    out = []
    for batch in loader:
        batch = {k: v.to(DEVICE) for k, v in batch.items() if k != 'labels'}
        with torch.autocast('cuda', dtype=torch.float16, enabled=DEVICE == 'cuda'):
            out.append(model(**batch).logits.float().cpu())
    return torch.cat(out)


use_amp = DEVICE == 'cuda'
opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
total_steps = EPOCHS * len(train_loader)
sched = get_linear_schedule_with_warmup(opt, int(WARMUP_FRAC * total_steps), total_steps)
scaler = torch.amp.GradScaler('cuda', enabled=use_amp)
loss_fn = torch.nn.CrossEntropyLoss(weight=class_weights)

best_f1, best_state, best_epoch, bad_epochs, history = -1.0, None, 0, 0, []
t0 = time.time()
for epoch in range(1, EPOCHS + 1):
    model.train()
    running = 0.0
    for step, batch in enumerate(train_loader, 1):
        batch = {k: v.to(DEVICE) for k, v in batch.items()}
        labels = batch.pop('labels')
        with torch.autocast('cuda', dtype=torch.float16, enabled=use_amp):
            loss = loss_fn(model(**batch).logits.float(), labels)
        opt.zero_grad(set_to_none=True)
        scaler.scale(loss).backward()
        scaler.unscale_(opt)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(opt)
        scaler.update()
        sched.step()
        running += loss.item()
        if step % 50 == 0 or step == len(train_loader):
            done = (epoch - 1) * len(train_loader) + step
            eta = (time.time() - t0) / done * (total_steps - done)
            print(f'epoch {epoch} step {step}/{len(train_loader)} loss {running / step:.4f} | ETA {eta / 60:.1f} min')

    val_pred = [EVENT_TYPES[i] for i in predict_logits(val_loader).argmax(1).tolist()]
    s = summary(val_df.y, val_pred)
    history.append({'epoch': epoch, 'train_loss': round(running / len(train_loader), 4), **s})
    print(f'== epoch {epoch}: validation {s}')
    if s['macro_f1'] > best_f1:
        best_f1, best_epoch, bad_epochs = s['macro_f1'], epoch, 0
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
    else:
        bad_epochs += 1
        if bad_epochs >= PATIENCE:
            print(f'No improvement for {PATIENCE} epochs — stopping early.')
            break

model.load_state_dict(best_state)
print(f'Best epoch {best_epoch} (validation macro-F1 {best_f1:.4f}) | training took {(time.time() - t0) / 60:.1f} min')

## 8. Calibrate confidence (temperature scaling)
Finds one temperature `T` so that softmax(logits / T) matches validation accuracy. Agent 4 uses the calibrated top-class probability as `risk_confidence`.

In [ ]:
val_logits = predict_logits(val_loader)
val_y = torch.tensor(val_df.y.map(LABEL2ID).to_numpy())

log_t = torch.zeros(1, requires_grad=True)
lbfgs = torch.optim.LBFGS([log_t], lr=0.05, max_iter=200)


def closure():
    lbfgs.zero_grad()
    loss = torch.nn.functional.cross_entropy(val_logits / log_t.exp(), val_y)
    loss.backward()
    return loss


lbfgs.step(closure)
TEMPERATURE = float(log_t.exp().clamp(0.5, 5.0))


def ece(probs, y, bins=10):
    conf, pred = probs.max(1)
    edges = torch.linspace(0, 1, bins + 1)
    total = 0.0
    for lo, hi in zip(edges[:-1], edges[1:]):
        m = (conf > lo) & (conf <= hi)
        if m.any():
            total += m.float().mean() * abs((pred[m] == y[m]).float().mean() - conf[m].mean())
    return float(total)


ece_before = ece(val_logits.softmax(1), val_y)
ece_after = ece((val_logits / TEMPERATURE).softmax(1), val_y)
print(f'temperature {TEMPERATURE:.3f} | calibration error (ECE) {ece_before:.3f} -> {ece_after:.3f}')

## 9. Validation report — DistilRoBERTa vs TF-IDF
Scored against the LLM's labels on the held-out 15%, so these numbers include the LLM's own mistakes.

In [ ]:
val_pred = [EVENT_TYPES[i] for i in val_logits.argmax(1).tolist()]
VAL_METRICS = {'distilroberta': summary(val_df.y, val_pred), 'tfidf': summary(val_df.y, tfidf_val_pred)}
print(pd.DataFrame(VAL_METRICS).T.to_string())
print('\nDistilRoBERTa per type:')
print(classification_report(val_df.y, val_pred, labels=EVENT_TYPES, digits=2, zero_division=0))
print('Confusion (rows = LLM label, columns = DistilRoBERTa):')
print(pd.crosstab(pd.Series(list(val_df.y), name='label'), pd.Series(val_pred, name='pred'))
      .reindex(index=EVENT_TYPES, columns=EVENT_TYPES, fill_value=0).to_string())

## 10. Gold evaluation — hand labels
Runs only when `gold_to_label.csv` has `gold_event_type` filled in. **These are the numbers to report.** Promotion rule: DistilRoBERTa becomes Agent 4's main model only if it beats TF-IDF on gold macro-F1.

In [ ]:
GOLD_METRICS = {}
gold = pd.read_csv(GOLD_CSV_PATH, dtype=str).fillna('') if GOLD_CSV_PATH else pd.DataFrame()
if len(gold) and 'gold_event_type' in gold:
    gold['gold_event_type'] = gold.gold_event_type.str.strip().str.lower()
    gold['gold_in_scope'] = gold.gold_in_scope.str.strip().str.lower()
    labelled = gold[gold.gold_event_type != '']
    bad = labelled[~labelled.gold_event_type.isin(EVENT_TYPES)]
    if len(bad):
        print(f'WARNING: {len(bad)} rows with an unknown event type, ignored:', bad.gold_event_type.unique().tolist())
    g = labelled[labelled.gold_event_type.isin(EVENT_TYPES) & (labelled.gold_in_scope != 'false')].copy()
else:
    g = pd.DataFrame()

if len(g) == 0:
    print('Gold set not labelled yet — skipping. Re-run this cell (and step 11) after hand-labelling.')
else:
    g['text'] = [event_text(t, b) for t, b in zip(g.title, g.body)]
    loader = make_loader(g.assign(y=g.gold_event_type), False)
    g['distilroberta'] = [EVENT_TYPES[i] for i in predict_logits(loader).argmax(1).tolist()]
    g['tfidf'] = tfidf.predict(g.text)
    cols = ['distilroberta', 'tfidf']
    if GOLD_KEY_PATH:
        key = pd.read_csv(GOLD_KEY_PATH, dtype=str)[['id', 'event_type']]
        g = g.merge(key.rename(columns={'event_type': 'qwen'}), on='id', how='left')
        g['qwen'] = g.qwen.replace(LEGACY_EVENT_TYPES).fillna('routine')
        cols.append('qwen')
    GOLD_METRICS = {c: summary(g.gold_event_type, g[c]) for c in cols}
    print(f'{len(g)} labelled in-scope gold articles\n')
    print(pd.DataFrame(GOLD_METRICS).T.to_string())
    print('\nDistilRoBERTa per type on gold:')
    print(classification_report(g.gold_event_type, g.distilroberta, labels=EVENT_TYPES, digits=2, zero_division=0))
    winner = 'distilroberta' if GOLD_METRICS['distilroberta']['macro_f1'] > GOLD_METRICS['tfidf']['macro_f1'] else 'tfidf'
    print(f'Promotion rule -> main model: {winner.upper()}')
    wrong = g[g.distilroberta != g.gold_event_type]
    print(f'\n{len(wrong)} DistilRoBERTa mistakes on gold:')
    for r in wrong.head(25).itertuples():
        print(f'  gold={r.gold_event_type:14s} pred={r.distilroberta:14s} | {(r.title or r.body)[:90]}')

## 11. Save the model
Writes `risk_event/` (weights, tokenizer, `label_config.json`) and zips it as `mapnai_risk_event_model.zip`.

In [ ]:
SAVE_DIR = os.path.join(OUT_DIR, 'risk_event')
shutil.rmtree(SAVE_DIR, ignore_errors=True)
model.save_pretrained(SAVE_DIR, safe_serialization=True)
tokenizer.save_pretrained(SAVE_DIR)
label_config = {
    'event_types': EVENT_TYPES,
    'event_severity': EVENT_SEVERITY,
    'temperature': round(TEMPERATURE, 4),
    'max_len': MAX_LEN,
    'max_body_chars': MAX_BODY_CHARS,
    'base_model': MODEL_NAME,
    'label_prompt_version': versions[-1] if versions else None,
    'fix_integrity': FIX_INTEGRITY,
    'trained_at': datetime.now(timezone.utc).isoformat(),
    'n_train': len(train_df), 'n_val': len(val_df), 'best_epoch': best_epoch,
    'metrics': {'validation': VAL_METRICS, 'gold': GOLD_METRICS, 'ece_after_calibration': round(ece_after, 4)},
    'history': history,
}
with open(os.path.join(SAVE_DIR, 'label_config.json'), 'w', encoding='utf-8') as f:
    json.dump(label_config, f, indent=2)

zip_path = shutil.make_archive(os.path.join(OUT_DIR, 'mapnai_risk_event_model'), 'zip', OUT_DIR, 'risk_event')
print('Saved', SAVE_DIR, '\nZip  ', zip_path, f'({os.path.getsize(zip_path) / 1e6:.0f} MB)')
print('Unzip into the repo so that models/risk_event/config.json exists.')
if ON_COLAB:
    from google.colab import files as colab_files
    colab_files.download(zip_path)